In [1]:
import urllib.request
import tarfile
import os
import duckdb
import json
from datetime import datetime
import pandas as pd

In [2]:
def formatar_timestamp(unix_timestamp):
    try:
        return datetime.fromtimestamp(int(unix_timestamp)).isoformat() + "Z"
    except:
        return datetime.utcnow().isoformat() + "Z"

def segundos_para_iso8601(segundos):
    m, s = divmod(int(segundos), 60)
    h, m = divmod(m, 60)
    iso = "PT"
    if h > 0: iso += f"{h}H"
    if m > 0: iso += f"{m}M"
    iso += f"{s}S"
    return iso

def obter_verbo_xapi(eventname):
    eventname = str(eventname).lower()
    if "viewed" in eventname: return "http://id.tincanapi.com/verb/viewed"
    if "submitted" in eventname: return "http://activitystrea.ms/schema/1.0/submit"
    if "completed" in eventname: return "http://adlnet.gov/expapi/verbs/completed"
    if "started" in eventname: return "http://adlnet.gov/expapi/verbs/attempted"
    return "http://id.tincanapi.com/verb/interacted"

In [3]:
# 1. URL corrigida com 'id_' no final do timestamp (padrão do Internet Archive para arquivo cru)
url = "https://web.archive.org/web/20210420235203id_/http://research.moodle.org/158/2/export.tar.gz"
arquivo_compactado = "export.tar.gz"
arquivo_verificacao = "export/mdl_logstore_standard_log.csv" 

def baixar_e_extrair_dados():
    if os.path.exists(arquivo_verificacao):
        print("✅ Os dados brutos já estão presentes na pasta. Pulando o download!")
        return

    print(f"📥 Iniciando o download do dataset gigante... (Isso pode demorar dependendo da conexão)")
    try:
        # 2. Criando um disfarce (User-Agent) para não sermos bloqueados
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'})
        
        # 3. Baixando o arquivo em "pedacinhos" para não travar a memória do computador
        with urllib.request.urlopen(req) as response, open(arquivo_compactado, 'wb') as out_file:
            while chunk := response.read(8192):
                out_file.write(chunk)
                
        print("📦 Download real concluído! Iniciando a extração dos arquivos...")

        # Extrai o .tar.gz
        with tarfile.open(arquivo_compactado, "r:gz") as tar:
            tar.extractall(path=".")
        
        print("🧹 Extração finalizada! Deletando o arquivo compactado para liberar espaço...")
        os.remove(arquivo_compactado)
        
        print("🚀 Tudo pronto! O dataset está descompactado e pronto para o Pandas.")
        
    except Exception as e:
        print(f"❌ Ocorreu um erro durante o processo: {e}")
        # Limpa o arquivo corrompido para não atrapalhar a próxima tentativa
        if os.path.exists(arquivo_compactado):
            os.remove(arquivo_compactado)

# Executa a função
baixar_e_extrair_dados()

✅ Os dados brutos já estão presentes na pasta. Pulando o download!


In [4]:
def extrair_e_cruzar_dados_duckdb():
    print("🦆 Iniciando DuckDB: Lendo e cruzando arquivos direto do disco...")
    
    pasta_dados = "export/"
    # A Query Mágica: Faz o JOIN, filtra e ordena usando C++ por baixo dos panos!
    query = f"""
        SELECT 
            logs.timecreated,
            logs.username,
            logs.eventname,
            logs.action,
            logs.component,
            logs.contextinstanceid,
            logs.courseid,
            notas.rawgrade,
            notas.rawgrademax,
            cm.section
        FROM read_csv_auto('{pasta_dados}mdl_logstore_standard_log.csv', ALL_VARCHAR=TRUE) AS logs
        
        -- Fazemos o cruzamento com as notas (LEFT JOIN)
        LEFT JOIN read_csv_auto('{pasta_dados}mdl_grade_grades_history.csv', ALL_VARCHAR=TRUE) AS notas
            ON logs.username = notas.username 
            AND logs.contextinstanceid = notas.itemid

        -- NOVO JOIN: Cruzando com os Módulos do Curso
        LEFT JOIN read_csv_auto('{pasta_dados}mdl_course_modules.csv', ALL_VARCHAR=TRUE) AS cm
            ON logs.contextinstanceid = cm.id

        -- Limpamos usuários inválidos ou sistema
        WHERE logs.username NOT IN ('0', '-1', '', 'nan')
          AND logs.username IS NOT NULL
          
        -- O DuckDB ordena tudo cronologicamente numa velocidade absurda
        ORDER BY CAST(logs.timecreated AS BIGINT) ASC
    """
    
    # Executa a query e converte o resultado limpo para um DataFrame leve do Pandas
    df_processado = duckdb.query(query).df()
    print(f"✅ Processamento concluído! O DuckDB preparou {len(df_processado)} linhas prontas para o xAPI.")
    
    return df_processado

# Executamos o motor
df_dados_limpos = extrair_e_cruzar_dados_duckdb()

🦆 Iniciando DuckDB: Lendo e cruzando arquivos direto do disco...
✅ Processamento concluído! O DuckDB preparou 2391762 linhas prontas para o xAPI.


In [5]:
def construir_data_lake_xapi(df):
    print("⚙️ Construindo Statements xAPI com Hierarquia Completa...")
    statements = []
    memoria_inicio = {}

    for _, row in df.iterrows():
        userid = str(row['username'])
        eventname = str(row['eventname'])
        timecreated = int(row['timecreated'])
        courseid = str(row['courseid'])
        instanceid = str(row['contextinstanceid'])
        component = str(row['component']).replace('mod_', '') # Limpa o nome (de 'mod_quiz' para 'quiz')
        sectionid = str(row['section'])
        
        chave_atividade = (userid, courseid, instanceid)

        if "started" in eventname or "viewed" in eventname:
            if chave_atividade not in memoria_inicio: 
                memoria_inicio[chave_atividade] = timecreated

        # === CONSTRUÇÃO DA ÁRVORE DE CONTEXTO ===
        parents = []
        
        # 1. Nível da Atividade
        tipo_atividade = "http://adlnet.gov/expapi/activities/assessment" if component == 'quiz' else f"http://adlnet.gov/expapi/activities/{component}"
        if instanceid != '0' and instanceid != 'nan':
            parents.append({
                "objectType": "Activity",
                "id": f"http://seumoodle.com/mod/{component}/view.php?id={instanceid}",
                "definition": {
                    "type": tipo_atividade,
                    "name": {"en": f"{component.capitalize()} (ID: {instanceid})"}
                }
            })

        # 2. Nível da Seção (Adiciona apenas se o DuckDB encontrou uma seção válida)
        if sectionid not in ['nan', 'None', '', '0']:
            parents.append({
                "objectType": "Activity",
                "id": f"http://seumoodle.com/course/section.php?id={sectionid}",
                "definition": {
                    "type": "http://id.tincanapi.com/activitytype/section",
                    "name": {"en": f"Seção ID {sectionid}"}
                }
            })

        # 3. Nível do Curso
        if courseid != '0' and courseid != 'nan':
            parents.append({
                "objectType": "Activity",
                "id": f"http://seumoodle.com/course/view.php?id={courseid}",
                "definition": {
                    "type": "https://w3id.org/xapi/cmi5/activitytype/course",
                    "name": {"en": f"Matéria ID {courseid}"}
                }
            })
        
        # Categoria do Sistema (LMS)
        contexto_atividades = {
            "category": [{
                "objectType": "Activity",
                "id": "http://seumoodle.com",
                "definition": {
                    "type": "http://id.tincanapi.com/activitytype/lms",
                    "name": {"en": "Moodle"}
                }
            }]
        }

        if len(parents) > 0:
            contexto_atividades["parent"] = parents
        # ========================================

        statement = {
            "actor": {"objectType": "Agent", "account": {"homePage": "http://seumoodle.com", "name": userid}},
            "verb": {
                "id": obter_verbo_xapi(eventname),
                "display": {"en": str(row['action'])}
            },
            "object": {
                "objectType": "Activity",
                "id": f"http://seumoodle.com/mod/{component}/view.php?id={instanceid}&course={courseid}",
                "definition": {"name": {"en": f"{component.capitalize()} (ID: {instanceid})"}}
            },
            "timestamp": formatar_timestamp(timecreated),
            
            # Injetando a Árvore de Contexto Completa
            "context": {
                "contextActivities": contexto_atividades
            }
        }

        # Lógica de Notas e Tempos
        if "submitted" in eventname or "completed" in eventname:
            statement["verb"]["id"] = "http://adlnet.gov/expapi/verbs/completed"
            statement["verb"]["display"] = {"en-US": "completed"}
            statement["result"] = {}
            
            if chave_atividade in memoria_inicio:
                tempo_gasto_segundos = timecreated - memoria_inicio[chave_atividade]
                if tempo_gasto_segundos >= 0:
                    statement["result"]["duration"] = segundos_para_iso8601(tempo_gasto_segundos)
                del memoria_inicio[chave_atividade]

            if pd.notna(row['rawgrade']) and pd.notna(row['rawgrademax']):
                statement["result"]["score"] = {
                    "raw": float(row['rawgrade']),
                    "max": float(row['rawgrademax'])
                }

            if not statement["result"]:
                del statement["result"]

        statements.append(statement)

    caminho_output = "xapi_statements_completos.json"
    print(f"💾 Salvando {len(statements)} statements unificados em '{caminho_output}'...")
    with open(caminho_output, "w", encoding="utf-8") as f:
        json.dump(statements, f, indent=4, ensure_ascii=False)
        
    print("🚀 Arquitetura Data Lake concluída! Estrutura de Contexto idêntica ao Logstore oficial.")

# Roda o montador final
construir_data_lake_xapi(df_dados_limpos)

⚙️ Construindo Statements xAPI com Hierarquia Completa...
💾 Salvando 2391762 statements unificados em 'xapi_statements_completos.json'...
🚀 Arquitetura Data Lake concluída! Estrutura de Contexto idêntica ao Logstore oficial.


In [6]:
import json

def inspecionar_data_lake():
    caminho_arquivo = "xapi_statements_completos.json"
    print(f"🔍 Abrindo o Data Lake '{caminho_arquivo}' para inspeção...\n")

    try:
        with open(caminho_arquivo, "r", encoding="utf-8") as f:
            statements = json.load(f)
            
        print(f"📊 Total de statements no arquivo: {len(statements):,}".replace(",", "."))
        
        print("\n" + "="*60)
        print("👀 EXEMPLOS DE INTERAÇÃO COMUM (Primeiros registros)")
        print("="*60)
        
        # Exibe os 2 primeiros registros
        for i in range(min(2, len(statements))):
            print(json.dumps(statements[i], indent=4, ensure_ascii=False))
            print("-" * 60)

        print("\n" + "="*60)
        print("🎯 EXEMPLO DE CONCLUSÃO (Com Nota e Tempo de Resposta)")
        print("="*60)
        
        # Busca o primeiro statement que contenha o bloco "result" (nota/tempo)
        statement_com_resultado = next((s for s in statements if "result" in s), None)
        
        if statement_com_resultado:
            print(json.dumps(statement_com_resultado, indent=4, ensure_ascii=False))
        else:
            print("Nenhum statement com 'result' encontrado. Verifique se as notas foram integradas.")
            
    except FileNotFoundError:
        print(f"❌ Erro: O arquivo '{caminho_arquivo}' não foi encontrado.")
    except Exception as e:
        print(f"❌ Erro ao ler o JSON: {e}")

# Executa a inspeção
inspecionar_data_lake()

🔍 Abrindo o Data Lake 'xapi_statements_completos.json' para inspeção...

📊 Total de statements no arquivo: 2.391.762

👀 EXEMPLOS DE INTERAÇÃO COMUM (Primeiros registros)
{
    "actor": {
        "objectType": "Agent",
        "account": {
            "homePage": "http://seumoodle.com",
            "name": "user6442803380426375169"
        }
    },
    "verb": {
        "id": "http://id.tincanapi.com/verb/interacted",
        "display": {
            "en": "loggedin"
        }
    },
    "object": {
        "objectType": "Activity",
        "id": "http://seumoodle.com/mod/core/view.php?id=0&course=0",
        "definition": {
            "name": {
                "en": "Core (ID: 0)"
            }
        }
    },
    "timestamp": "2014-12-04T12:32:49Z",
    "context": {
        "contextActivities": {
            "category": [
                {
                    "objectType": "Activity",
                    "id": "http://seumoodle.com",
                    "definition": {
             

In [7]:
import shutil
import os

# Caminhos absolutos exatos (o 'r' na frente evita problemas com as barras do Windows)
origem = r'C:\Users\Danilus04\Documents\Faculdade\TCC\learning-analytics\transformador\xapi_statements_completos.json'
destino_pasta = r'C:\Users\Danilus04\Documents\Faculdade\TCC\learning-analytics\gerador-de-medida\data'
destino_arquivo = os.path.join(destino_pasta, 'xapi_statements_completos.json')

def mover_resultado():
    # Verificar se o arquivo de origem existe
    if not os.path.exists(origem):
        print(f"❌ Erro: O arquivo de origem não foi encontrado em:\n{origem}")
        return

    # Criar a pasta de destino se ela não existir
    if not os.path.exists(destino_pasta):
        os.makedirs(destino_pasta)
        print(f"📁 Pasta de destino criada: {destino_pasta}")

    try:
        # Mover o arquivo
        shutil.move(origem, destino_arquivo)
        print(f"✅ Sucesso! Arquivo movido para:\n{destino_arquivo}")
    except Exception as e:
        print(f"❌ Erro ao mover o arquivo: {e}")

mover_resultado()

✅ Sucesso! Arquivo movido para:
C:\Users\Danilus04\Documents\Faculdade\TCC\learning-analytics\gerador-de-medida\data\xapi_statements_completos.json
